In [4]:
import json

def write_jsonl(data, file_path):
    """
    Write a list of dictionaries to a JSONL file.

    :param data: List of dictionaries to write to the file
    :param file_path: Path to the JSONL file
    """
    with open(file_path, 'w') as file:
        for entry in data:
            json_line = json.dumps(entry)
            file.write(json_line + '\n')

def read_jsonl(file_path):
    """
    Read a JSONL file and return a list of dictionaries.

    :param file_path: Path to the JSONL file
    :return: List of dictionaries
    """
    with open(file_path, 'r') as file:
        return [json.loads(line) for line in file]
    
import csv
def read_tsv(file_path):
    with open(file_path, 'r') as file:
        return [line for line in csv.reader(file, delimiter='\t')]
    
def write_tsv(data, file_path):
    with open(file_path, 'w') as file:
        writer = csv.writer(file, delimiter='\t')
        writer.writerows(data)

In [3]:
import os
import json
import pandas as pd

# Define the directory containing the JSONL files
input_dir = 'outputs'
output_dir = 'outputs_csv'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Iterate over each file in the directory
for filename in os.listdir(input_dir):
    if filename.endswith('.jsonl'):
        jsonl_path = os.path.join(input_dir, filename)
        csv_path = os.path.join(output_dir, filename.replace('.jsonl', '.csv'))
        
        # Read the JSONL file
        with open(jsonl_path, 'r') as jsonl_file:
            data = [json.loads(line) for line in jsonl_file]
        
        # Convert to DataFrame
        df = pd.DataFrame(data)
        
        # Write to CSV
        df.to_csv(csv_path, index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'outputs'

In [ ]:
import json
from tqdm import tqdm

# Read the JSONL file
with open('qampari_data/dev_data.jsonl', 'r') as jsonl_file:
    data = [json.loads(line) for line in jsonl_file]

# Modify the data format and field names
modified_data = []
for item in tqdm(data):
    modified_item = {
        'question': item['question_text'],
        'input': item['question_text'],
        'id': item['qid']
    }
    modified_data.append(modified_item)

# Write the modified data to JSON file
with open('qampari_data/nqformat_data/qampari_dev.json', 'w') as json_file:
    json.dump(modified_data, json_file)


: 

In [ ]:
import csv
from pathlib import Path
import random

## Observe Retrieval Outputs

retrieve_dir = Path('/scratch/cluster/hungting/projects/Multi_Answer/contriever')
data_dir = Path('/scratch/cluster/hungting/projects/diverse_response/data/qampari_data/2nd_stage')
retriever = 'contriever'

# columns needed: question, answers (entity names for QUEST), retrieved documents
retrieved_outputs = read_jsonl(retrieve_dir / 'outputs/contriever_msmarco_wiki_2nd_stage/qampari_dev_2nd_stage_comb.json')
data = read_jsonl(data_dir / 'dev_data.jsonl')
assert len(retrieved_outputs) == len(data)

outputs = [['question', 'answers', 'retrieved_document_1', 'retrieved_document_2', 'retrieved_document_3', 'retrieved_document_4', 'retrieved_document_5']]
for inst, retrieved in zip(data, retrieved_outputs):
    answers = json.dumps(inst['positive_ctxs'])
    retrieved_documents = [d['title'] + ' || ' + d['text'] for d in retrieved['ctxs']]
    outputs.append([retrieved['question'], answers] + retrieved_documents)    
    
write_tsv(outputs, Path('../data/csvs') / f'qampari_{retriever}_dev_2nd_stage_comb.tsv')